# Exp.04B — PGR²M Refine Transformer Recovery + 150k Training

Exp.04A has completed and its **best-validation-FID Base Transformer checkpoint** is now frozen.
This notebook reconstructs the missing Refine-Transformer training path and then performs the
150k Refine Transformer training required before the final conditioning-intervention study.

## Frozen upstream stack

- **Tokenizer:** Exp.02 matched default, `tau=0.1`, entropy weight `0.01`, seed 123
- **Base Transformer:** Exp.04A, 300k training completed
- **Selected Base checkpoint:** best validation FID at **290k**

## Public-release gaps

The public PGR²M repository does not contain `dataset_RTM_train.py`, `dataset_RTM_eval.py`, or
`configs/exp_rt2m/losses.yaml`. This notebook therefore creates **contract-compatible operational
reconstructions** from the released training/evaluation code. These are not claimed to be the
unreleased private originals.

The reconstructed RTM loaders reuse the already validated TM loaders and obtain RVQ residual
targets online from the **frozen tokenizer**. Because that tokenizer is a CUDA model, the RTM
loaders deliberately use `num_workers=0` so CUDA is never touched inside DataLoader worker
processes.

## Refine Transformer protocol

The architecture/training settings follow the released README: 6 layers, 8 heads, embedding 1024,
batch size 64, 150k iterations, LR 2e-4, LR milestone 50k, 5k warm-up, `pkeep=1.0`, residual
masking probability 0, dropout 0.2, shared residual embedding/projection weights, frozen
pretrained pose-code embedding, keyword conditioning, and seed 123.

Legacy 3-D TensorBoard rendering is disabled (`draw=False`) because it is incompatible with the
current Matplotlib version. **Quantitative validation and best-FID checkpoint selection remain
unchanged.**


In [ ]:
from pathlib import Path
import os, sys, json, re, time, hashlib, subprocess, shlex

PGR = Path('.')
PY = Path('venv/bin/python')
PRE = PGR / 'revision_preflight'
LOGDIR = PGR / 'revision_logs' / 'exp04b_refine'
RESULTS = PGR / 'revision_results' / 'exp04b_refine'
OUT_ROOT = PGR / 'exp_result_revision_refine_transformer'

BASE_AUDIT = PGR / 'revision_results' / 'exp04a_base' / 'exp04a_base_completion_audit.json'
TRAIN_SRC = PGR / 'train_refine_trans.py'
RTM_TRAIN = PGR / 'dataset' / 'dataset_RTM_train.py'
RTM_EVAL = PGR / 'dataset' / 'dataset_RTM_eval.py'
LOSS_CFG = PRE / 'exp04b_rt2m_losses.yaml'
WRAPPER = PRE / 'train_refine_trans_exp04b_compat.py'

LOG = LOGDIR / 'exp04b_refine_train.log'
PIDFILE = LOGDIR / 'exp04b_refine.pid'
LAUNCH = LOGDIR / 'exp04b_refine_launch.json'
FREEZE = RESULTS / 'exp04b_refine_config_freeze.json'
AUDIT = RESULTS / 'exp04b_refine_completion_audit.json'

for d in [PRE, LOGDIR, RESULTS, OUT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

print('=== EXP.04B ENVIRONMENT ===')
print('Python  :', sys.executable)
print('Expected:', PY)
assert PGR.exists(), PGR
assert PY.exists(), PY
assert TRAIN_SRC.exists(), TRAIN_SRC
assert BASE_AUDIT.exists(), BASE_AUDIT

base_audit = json.loads(BASE_AUDIT.read_text())
BASE_DIR = Path(base_audit['run_dir'])
BASE_CKPT = Path(base_audit['best_fid_checkpoint'])
TOKENIZER = Path(base_audit['tokenizer_dir'])
TOKENIZER_CKPT = TOKENIZER / 'net_best_fid.pth'

for pth in [BASE_DIR/'arguments.yaml', BASE_CKPT, TOKENIZER/'arguments.yaml', TOKENIZER_CKPT,
            PGR/'dataset'/'dataset_TM_train.py', PGR/'dataset'/'dataset_TM_eval.py']:
    assert pth.exists(), pth

print()
print('=== FROZEN MATCHED STACK ===')
print('Tokenizer :', TOKENIZER)
print('Tokenizer SHA:', sha256(TOKENIZER_CKPT))
print('Base      :', BASE_DIR)
print('Base ckpt :', BASE_CKPT)
print('Base iter :', base_audit.get('checkpoint_nb_iter'))
print('Base SHA  :', sha256(BASE_CKPT))
assert int(base_audit['checkpoint_nb_iter']) == 290000, base_audit
print('[OK] Exp.04A best-FID Base Transformer resolved.')


In [ ]:
# Reconstruct the missing Refine-Transformer CE loss config.
# The released CEWithLogitsLossWrapper consumes a single `loss_ce` weight.
LOSS_CFG.write_text("""loss_wrappers:
  - type: CEWithLogitsLossWrapper
    name: ce
    params:
      weight:
        loss_ce: 1.0
      use_in_loss: true
""")
print(LOSS_CFG.read_text())
print('[OK] Refine loss config reconstructed:', LOSS_CFG)


In [ ]:
# Reconstruct dataset_RTM_train.py.
# It wraps the validated dataset_TM_train loader and adds frozen-tokenizer RVQ targets.
rtm_train_text = r'''import inspect
import torch
from dataset import dataset_TM_train as _tm


def _call_supported(fn, kwargs):
    sig = inspect.signature(fn)
    return fn(**{k:v for k,v in kwargs.items() if k in sig.parameters})


def _augment(pg_tokenizer, batch, rvq_codebook_size, num_quantizer):
    caption, m_tokens, m_tokens_len, keyword_embeddings, gt_motion, m_length = batch[:6]

    # m_tokens is [B,51,394]. Refine training later removes the last two time slots,
    # leaving 49 positions. The tokenizer itself receives only the 392 pose-code dims.
    pose_codes = m_tokens[:, :-2, :-2].float().cuda(non_blocking=False)
    n_tok = pose_codes.shape[1]

    # Tokenizer temporal stride is x4 for down_t=2. Use exactly 49*4=196 frames.
    motion = gt_motion[:, :n_tok*4, :].float().cuda(non_blocking=False)

    with torch.no_grad():
        _, residual_onehot = pg_tokenizer.tokenize(
            motion=motion,
            code_indices=pose_codes,
            return_one_hot=True,
        )

    assert residual_onehot.ndim == 4, residual_onehot.shape
    assert residual_onehot.shape[1] == n_tok, (residual_onehot.shape, n_tok)
    assert residual_onehot.shape[2] == num_quantizer, residual_onehot.shape
    assert residual_onehot.shape[3] == rvq_codebook_size, residual_onehot.shape

    # train_refine_trans.py slices m_res_tokens_new[:, :-2], so append two dummy slots.
    pad = torch.zeros(
        residual_onehot.shape[0], 2, num_quantizer, rvq_codebook_size,
        device=residual_onehot.device, dtype=residual_onehot.dtype
    )
    residual_new = torch.cat([residual_onehot, pad], dim=1)

    return (caption, m_tokens, m_tokens_len, keyword_embeddings,
            gt_motion, m_length, residual_new)


class _RTMLoader:
    def __init__(self, base_loader, pg_tokenizer, rvq_codebook_size, num_quantizer):
        self.base_loader = base_loader
        self.dataset = base_loader.dataset
        self.pg_tokenizer = pg_tokenizer
        self.rvq_codebook_size = rvq_codebook_size
        self.num_quantizer = num_quantizer

    def __len__(self):
        return len(self.base_loader)

    def __iter__(self):
        for batch in self.base_loader:
            yield _augment(self.pg_tokenizer, batch,
                           self.rvq_codebook_size, self.num_quantizer)


def DATALoader(dataset_name, batch_size, codebook_size, pg_tokenizer,
               unit_length=4, rvq_codebook_size=512, num_quantizer=6,
               num_workers=0, codes_folder_name='codes', cache_file_name=None,
               use_keywords=False):
    # Force workers=0: residual target generation uses the CUDA tokenizer.
    base = _call_supported(_tm.DATALoader, dict(
        dataset_name=dataset_name,
        batch_size=batch_size,
        codebook_size=codebook_size,
        unit_length=unit_length,
        num_workers=0,
        use_keywords=use_keywords,
    ))
    return _RTMLoader(base, pg_tokenizer, rvq_codebook_size, num_quantizer)


def cycle(iterable):
    while True:
        for x in iterable:
            yield x
'''

RTM_TRAIN.write_text(rtm_train_text)
compile(RTM_TRAIN.read_text(), str(RTM_TRAIN), 'exec')
print('[OK] Wrote:', RTM_TRAIN)


In [ ]:
# Reconstruct dataset_RTM_eval.py.
# It wraps the validated dataset_TM_eval loader and supplies the extra pose/residual
# tensors required by train_refine_trans.py's validation-loss path.
rtm_eval_text = r'''import inspect
import torch
from dataset import dataset_TM_eval as _tm


def _call_supported(fn, kwargs):
    sig = inspect.signature(fn)
    return fn(**{k:v for k,v in kwargs.items() if k in sig.parameters})


def _augment(pg_tokenizer, batch, codebook_size, rvq_codebook_size, num_quantizer):
    # Validated TM evaluator contract: the first ten fields are standard.
    word_embeddings, pos_one_hots, clip_text, sent_len, pose, m_length, token, name, indices, keyword_embeddings = batch[:10]

    # The reconstructed TM evaluator returns frame-level 392-D pose codes over 196 frames.
    # Downsample by 4 to match the tokenizer latent length (49 positions).
    pose_codes = indices[:, :196:4, :codebook_size].float().cuda(non_blocking=False)
    n_tok = pose_codes.shape[1]
    motion = pose[:, :n_tok*4, :].float().cuda(non_blocking=False)

    with torch.no_grad():
        _, residual_onehot = pg_tokenizer.tokenize(
            motion=motion,
            code_indices=pose_codes,
            return_one_hot=True,
        )

    assert residual_onehot.ndim == 4, residual_onehot.shape
    assert residual_onehot.shape[1] == n_tok, (residual_onehot.shape, n_tok)
    assert residual_onehot.shape[2] == num_quantizer, residual_onehot.shape
    assert residual_onehot.shape[3] == rvq_codebook_size, residual_onehot.shape

    # Refine pose embedding expects num_vq+2=394 dimensions.
    pose_new = torch.zeros(
        pose_codes.shape[0], n_tok+2, codebook_size+2,
        device=pose_codes.device, dtype=pose_codes.dtype
    )
    pose_new[:, :n_tok, :codebook_size] = pose_codes

    residual_new = torch.zeros(
        residual_onehot.shape[0], n_tok+2, num_quantizer, rvq_codebook_size,
        device=residual_onehot.device, dtype=residual_onehot.dtype
    )
    residual_new[:, :n_tok] = residual_onehot

    return (word_embeddings, pos_one_hots, clip_text, sent_len, pose, m_length,
            token, name, indices, keyword_embeddings, pose_new, residual_new)


class _RTMEvalLoader:
    def __init__(self, base_loader, pg_tokenizer, codebook_size,
                 rvq_codebook_size, num_quantizer):
        self.base_loader = base_loader
        self.dataset = base_loader.dataset
        self.pg_tokenizer = pg_tokenizer
        self.codebook_size = codebook_size
        self.rvq_codebook_size = rvq_codebook_size
        self.num_quantizer = num_quantizer

    def __len__(self):
        return len(self.base_loader)

    def __iter__(self):
        for batch in self.base_loader:
            yield _augment(self.pg_tokenizer, batch, self.codebook_size,
                           self.rvq_codebook_size, self.num_quantizer)


def DATALoader(dataset_name, is_test, batch_size, w_vectorizer,
               codebook_size=392, pg_tokenizer=None, rvq_codebook_size=512,
               num_quantizer=6, num_workers=0, codes_folder_name='codes',
               cache_file_name=None, use_keywords=False):
    base = _call_supported(_tm.DATALoader, dict(
        dataset_name=dataset_name,
        is_test=is_test,
        batch_size=batch_size,
        w_vectorizer=w_vectorizer,
        codebook_size=codebook_size,
        num_workers=0,
        unit_length=4,
        use_keywords=use_keywords,
    ))
    return _RTMEvalLoader(base, pg_tokenizer, codebook_size,
                          rvq_codebook_size, num_quantizer)


def get_subsamples_loader(*args, **kwargs):
    raise NotImplementedError(
        'Qualitative drawing is disabled in Exp.04B; this unreleased helper is not used.'
    )
'''

RTM_EVAL.write_text(rtm_eval_text)
compile(RTM_EVAL.read_text(), str(RTM_EVAL), 'exec')
print('[OK] Wrote:', RTM_EVAL)


In [ ]:
# Create a small, fast loader-contract smoke script.
# This catches tensor/shape mismatches BEFORE the expensive end-to-end Refine validation.
CONTRACT = PRE / 'exp04b_rtm_contract_smoke.py'

contract_text = r'''import os, sys, json, argparse
from pathlib import Path
import numpy as np
import torch

PGR = Path('.')
os.chdir(PGR)
sys.path.insert(0, str(PGR))

if not hasattr(np, 'float'): np.float = float
if not hasattr(np, 'int'): np.int = int
if not hasattr(np, 'bool'): np.bool = bool

from models.pg_tokenizer import PoseGuidedTokenizer
from dataset import dataset_RTM_train, dataset_RTM_eval
from utils.word_vectorizer import WordVectorizer

base_audit = json.loads((PGR/'revision_results/exp04a_base/exp04a_base_completion_audit.json').read_text())
tok_dir = Path(base_audit['tokenizer_dir'])
args_dict = json.loads((tok_dir/'arguments.yaml').read_text())
a = argparse.Namespace(**args_dict)

net = PoseGuidedTokenizer(
    a, a.nb_code, a.code_dim, a.output_emb_width, a.down_t, a.stride_t,
    a.width, a.depth, a.dilation_growth_rate, a.vq_act, a.vq_norm,
    num_quantizers=a.rvq_num_quantizers,
    shared_codebook=a.rvq_shared_codebook,
    quantize_dropout_prob=a.rvq_quantize_dropout_prob,
    quantize_dropout_cutoff_index=a.rvq_quantize_dropout_cutoff_index,
    rvq_nb_code=a.rvq_nb_code,
    mu=a.rvq_mu,
    residual_ratio=a.rvq_residual_ratio,
    vq_loss_beta=a.rvq_vq_loss_beta,
    quantizer_type=a.rvq_quantizer_type,
    params_soft_ent_loss=a.params_soft_ent_loss,
    use_ema=(not a.unuse_ema),
    init_method=a.rvq_init_method,
)
ck = torch.load(tok_dir/'net_best_fid.pth', map_location='cpu')
net.load_state_dict(ck['net'], strict=True)
net.eval().cuda()
for p in net.parameters(): p.requires_grad=False

tr = dataset_RTM_train.DATALoader('t2m', 2, a.nb_code, pg_tokenizer=net,
    unit_length=4, rvq_codebook_size=a.rvq_nb_code,
    num_quantizer=a.rvq_num_quantizers, num_workers=0, use_keywords=True)
b = next(iter(tr))
print('[RUN] train fields:', len(b))
print('[RUN] train pose:', tuple(b[1].shape), 'res:', tuple(b[6].shape), 'motion:', tuple(b[4].shape))
assert len(b) == 7
assert tuple(b[1].shape[1:]) == (51, 394)
assert tuple(b[6].shape[1:]) == (51, 6, 512)
assert tuple(b[4].shape[1:]) == (200, 263)
assert torch.isfinite(b[6]).all()

wv = WordVectorizer('./glove', 'our_vab')
ev = dataset_RTM_eval.DATALoader('t2m', False, 2, wv,
    codebook_size=a.nb_code, pg_tokenizer=net,
    rvq_codebook_size=a.rvq_nb_code, num_quantizer=a.rvq_num_quantizers,
    num_workers=0, use_keywords=True)
e = next(iter(ev))
print('[RUN] eval fields:', len(e))
print('[RUN] eval pose target:', tuple(e[10].shape), 'res:', tuple(e[11].shape), 'motion:', tuple(e[4].shape))
assert len(e) == 12
assert tuple(e[10].shape[1:]) == (51, 394)
assert tuple(e[11].shape[1:]) == (51, 6, 512)
assert tuple(e[4].shape[1:]) == (196, 263)
assert torch.isfinite(e[11]).all()
print('[OK] Exp.04B RTM loader contract passed.')
'''
CONTRACT.write_text(contract_text)
compile(CONTRACT.read_text(), str(CONTRACT), 'exec')

p = subprocess.run([str(PY), '-u', str(CONTRACT)], cwd=PGR,
                   text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                   timeout=3600)
print(p.stdout)
if p.returncode != 0:
    raise RuntimeError('Exp.04B RTM loader contract smoke failed.')


In [ ]:
# Create production compatibility wrapper.
# - pinned OpenAI CLIP checkout
# - NumPy 2.x aliases
# - missing TM get_subsamples_loader import shim
# - force Refine validation draw=False (quantitative validation remains intact)
CLIP_ROOT = PRE / 'third_party' / 'openai_clip_dcba3cb2e282'
assert CLIP_ROOT.exists(), CLIP_ROOT

wrapper_text = r'''import os, sys, runpy
from pathlib import Path
import numpy as np

PGR = Path('.')
CLIP_ROOT = PGR/'revision_preflight/third_party/openai_clip_dcba3cb2e282'
os.chdir(PGR)
sys.path.insert(0, str(PGR))
sys.path.insert(0, str(CLIP_ROOT))

if not hasattr(np, 'float'): np.float = float
if not hasattr(np, 'int'): np.int = int
if not hasattr(np, 'bool'): np.bool = bool

import ftfy
import clip
print('[OK] CLIP:', clip.__file__)

def _unused_subsamples(*args, **kwargs):
    raise NotImplementedError('Qualitative drawing is disabled in Exp.04B.')

import dataset.dataset_TM_eval as _tm_eval
if not hasattr(_tm_eval, 'get_subsamples_loader'):
    _tm_eval.get_subsamples_loader = _unused_subsamples
    print('[OK] Added dataset_TM_eval.get_subsamples_loader import shim')

import dataset.dataset_RTM_eval as _rtm_eval
import utils.eval_trans as _eval_trans
_orig = _eval_trans.evaluation_residual_transformer

def _eval_no_draw(*args, **kwargs):
    kwargs['draw'] = False
    return _orig(*args, **kwargs)

_eval_trans.evaluation_residual_transformer = _eval_no_draw
print('[OK] Forced evaluation_residual_transformer(draw=False)')

runpy.run_path(str(PGR/'train_refine_trans.py'), run_name='__main__')
'''

WRAPPER.write_text(wrapper_text)
compile(WRAPPER.read_text(), str(WRAPPER), 'exec')
print('[OK] Wrapper:', WRAPPER)


In [ ]:
# Freeze the production Refine configuration.
# num_workers=0 is intentional for this loader reconstruction because residual targets
# are generated by the frozen CUDA tokenizer in the main process.
EXP_NAME = 'exp_refine_trans_revision_seed123'
COMMON = [
    '--exp-name', EXP_NAME,
    '--batch-size', '64',
    '--num-layers', '6',
    '--nb-code', '392',
    '--n-head-gpt', '8',
    '--block-size', '62',
    '--ff-rate', '4',
    '--embed-dim-gpt', '1024',
    '--out-dir', str(OUT_ROOT),
    '--total-iter', '150000',
    '--lr-scheduler', '50000',
    '--lr', '0.0002',
    '--dataname', 't2m',
    '--down-t', '2',
    '--depth', '3',
    '--eval-iter', '10000',
    '--eval-loss-iter', '1000',
    '--pkeep', '1.0',
    '--num_workers', '0',
    '--dilation-growth-rate', '3',
    '--output-emb-width', '392',
    '--val-shuffle',
    '--loss-cfg-path', str(LOSS_CFG),
    '--log-cat-right-num',
    '--min-sampling-prob', '0.1',
    '--masking-prob', '0.0',
    '--drop-out-rate', '0.2',
    '--share-weight',
    '--warm-up-iter', '5000',
    '--start-warm-up',
    '--freeze-pose-code-emb',
    '--load-pretrained-pose-code-emb',
    '--dec-checkpoint-folder', str(TOKENIZER),
    '--t2m-checkpoint-folder', str(BASE_DIR),
    '--codes-folder-name', 'codes',
    '--use-keywords',
    '--seed', '123',
]

freeze = {
    'experiment': 'Exp04B Refine Transformer recovery',
    'tokenizer_dir': str(TOKENIZER),
    'tokenizer_sha256': sha256(TOKENIZER_CKPT),
    'base_dir': str(BASE_DIR),
    'base_checkpoint': str(BASE_CKPT),
    'base_checkpoint_nb_iter': int(base_audit['checkpoint_nb_iter']),
    'base_sha256': sha256(BASE_CKPT),
    'train_source_sha256': sha256(TRAIN_SRC),
    'rtm_train_sha256': sha256(RTM_TRAIN),
    'rtm_eval_sha256': sha256(RTM_EVAL),
    'loss_cfg_sha256': sha256(LOSS_CFG),
    'loader_provenance': 'contract-compatible reconstruction; online frozen-tokenizer residual targets',
    'loss_provenance': 'reconstructed from released CEWithLogitsLossWrapper contract',
    'num_workers_note': 'forced to 0 because reconstructed RTM loaders invoke CUDA tokenizer in main process',
    'args': COMMON,
}
FREEZE.write_text(json.dumps(freeze, indent=2))
print('[OK] Freeze:', FREEZE)
print('Command:')
print(' '.join(shlex.quote(x) for x in [str(PY), '-u', str(WRAPPER), *COMMON]))


In [ ]:
# End-to-end production-architecture smoke.
# This performs the mandatory initial quantitative generation validation and one
# Refine update. It may take several minutes. DO NOT launch 150k until this passes.
SMOKE_OUT = PRE / 'exp04b_refine_smoke_out'
SMOKE_LOG = LOGDIR / 'exp04b_refine_smoke.log'

SMOKE = [
    '--exp-name', 'exp04b_refine_smoke',
    '--batch-size', '64',
    '--num-layers', '6',
    '--nb-code', '392',
    '--n-head-gpt', '8',
    '--block-size', '62',
    '--ff-rate', '4',
    '--embed-dim-gpt', '1024',
    '--out-dir', str(SMOKE_OUT),
    '--total-iter', '1',
    '--lr-scheduler', '50000',
    '--lr', '0.0002',
    '--dataname', 't2m',
    '--down-t', '2',
    '--depth', '3',
    '--eval-iter', '999999',
    '--eval-loss-iter', '999999',
    '--pkeep', '1.0',
    '--num_workers', '0',
    '--dilation-growth-rate', '3',
    '--output-emb-width', '392',
    '--val-shuffle',
    '--loss-cfg-path', str(LOSS_CFG),
    '--log-cat-right-num',
    '--min-sampling-prob', '0.1',
    '--masking-prob', '0.0',
    '--drop-out-rate', '0.2',
    '--share-weight',
    '--warm-up-iter', '5000',
    '--start-warm-up',
    '--freeze-pose-code-emb',
    '--load-pretrained-pose-code-emb',
    '--dec-checkpoint-folder', str(TOKENIZER),
    '--t2m-checkpoint-folder', str(BASE_DIR),
    '--codes-folder-name', 'codes',
    '--use-keywords',
    '--seed', '123',
]

cmd = [str(PY), '-u', str(WRAPPER), *SMOKE]
print('=== EXP.04B REFINE SMOKE ===')
print(' '.join(shlex.quote(x) for x in cmd))

t0 = time.time()
with open(SMOKE_LOG, 'w') as f:
    proc = subprocess.run(cmd, cwd=PGR, stdout=f, stderr=subprocess.STDOUT, timeout=21600)
elapsed = (time.time()-t0)/60

txt = SMOKE_LOG.read_text(errors='replace')
print('returncode:', proc.returncode)
print(f'elapsed: {elapsed:.1f} min')
print(chr(10).join(txt.splitlines()[-220:]))
if proc.returncode != 0:
    raise RuntimeError(f'Refine Transformer smoke failed: {SMOKE_LOG}')

required = ['loading transformer checkpoint', 'loading decoder checkpoint', 'Model Main Training Start From']
missing = [m for m in required if m.lower() not in txt.lower()]
if missing:
    raise RuntimeError(f'Smoke returned 0 but markers missing: {missing}')
print('[OK] Exp.04B end-to-end smoke passed.')


In [ ]:
# Launch 150k Refine Transformer training.
# Run ONLY after the previous cell prints the [OK] smoke message.
def pid_alive(pid):
    if not pid: return False
    try:
        os.kill(pid, 0); return True
    except Exception:
        return False

old_pid=None
if PIDFILE.exists():
    try: old_pid=int(PIDFILE.read_text().strip())
    except Exception: pass

RUN_ROOT = OUT_ROOT / EXP_NAME
log_text = LOG.read_text(errors='replace') if LOG.exists() else ''
already_done = ('Train. Iter 150000:' in log_text)

if already_done and list(RUN_ROOT.rglob('net_best_fid.pth')):
    print('=== EXP.04B ALREADY COMPLETE ===')
elif old_pid and pid_alive(old_pid):
    print('=== EXP.04B ALREADY RUNNING ===')
    print('PID:', old_pid)
else:
    cmd=[str(PY), '-u', str(WRAPPER), *COMMON]
    handle=open(LOG, 'a')
    p=subprocess.Popen(cmd, cwd=PGR, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
    PIDFILE.write_text(str(p.pid))
    LAUNCH.write_text(json.dumps({
        'pid':p.pid, 'log':str(LOG), 'output_root':str(OUT_ROOT),
        'exp_name':EXP_NAME, 'command':cmd, 'launched_at':time.time()
    }, indent=2))
    time.sleep(1)
    print('=== EXP.04B REFINE TRAIN LAUNCH ===')
    print('PID   :', p.pid)
    print('Alive :', pid_alive(p.pid))
    print('Log   :', LOG)
    print('Output:', RUN_ROOT)


In [ ]:
# Read-only progress monitor. Safe to rerun.
def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

def max_train_iter(path):
    if not path.exists():
        return None
    pat = re.compile(r'Train\. Iter\s+(\d+):\s+Total_loss')
    last = None
    for line in path.read_text(errors='replace').splitlines():
        m = pat.search(line)
        if m:
            last = max(last or 0, int(m.group(1)))
    return last

pid = None
if PIDFILE.exists():
    try:
        pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass
it = max_train_iter(LOG)
pct = 100.0*it/150000 if it is not None else 0.0
print('=== EXP.04B REFINE STATUS ===')
print('PID   :', pid)
print('Alive :', pid_alive(pid) if pid else False)
print('Iter  :', it)
print(f'Progress: {pct:.2f}%')
print('Log   :', LOG)
print()
print('=== LOG TAIL ===')
if LOG.exists():
    print(chr(10).join(LOG.read_text(errors='replace').splitlines()[-100:]))


In [ ]:
# Completion / selected-checkpoint audit. Run after training reaches 150k.
import torch

txt = LOG.read_text(errors='replace') if LOG.exists() else ''
if 'Train. Iter 150000:' not in txt:
    raise RuntimeError('Refine training has not reached 150000 yet.')

RUN_ROOT = OUT_ROOT / EXP_NAME
ckpts = sorted(RUN_ROOT.rglob('net_best_fid.pth'), key=lambda p: p.stat().st_mtime)
if not ckpts:
    raise RuntimeError(f'No net_best_fid.pth under {RUN_ROOT}')
BEST = ckpts[-1]
RUN_DIR = BEST.parent
ARGS = RUN_DIR / 'arguments.yaml'
ck = torch.load(BEST, map_location='cpu')
ck_iter = ck.get('nb_iter')

pat = re.compile(r'Eva\. Iter\s+(\d+)\s*:, FID\.\s*([0-9.eE+-]+)')
vals = {}
for m in pat.finditer(txt):
    vals[int(m.group(1))] = float(m.group(2))
if not vals:
    raise RuntimeError('No validation FID entries parsed from Refine log.')
parsed_iter = min(vals, key=vals.get)
parsed_fid = vals[parsed_iter]

audit = {
    'final_train_iter': 150000,
    'run_dir': str(RUN_DIR),
    'best_fid_checkpoint': str(BEST),
    'checkpoint_nb_iter': int(ck_iter) if ck_iter is not None else None,
    'parsed_best_val_iter': int(parsed_iter),
    'parsed_best_val_fid': float(parsed_fid),
    'checkpoint_sha256': sha256(BEST),
    'arguments_sha256': sha256(ARGS),
    'base_dir': str(BASE_DIR),
    'base_checkpoint_sha256': sha256(BASE_CKPT),
    'tokenizer_dir': str(TOKENIZER),
    'tokenizer_checkpoint_sha256': sha256(TOKENIZER_CKPT),
}
AUDIT.write_text(json.dumps(audit, indent=2))

print('=== EXP.04B COMPLETION AUDIT ===')
for k, v in audit.items():
    print(f'{k}: {v}')
if ck_iter is not None and int(ck_iter) == int(parsed_iter):
    print('[OK] Best-FID checkpoint and parsed minimum agree.')
else:
    print('[WARN] Checkpoint iteration and parsed minimum differ; inspect before Exp.04C.')
print()
print('Next: Exp.04C fixed-checkpoint conditioning interventions')
print('full / no_text / no_pose / no_stage / no_history')
